# HMC Run Analysis — Interactive
Select a run, adjust burn-in, and choose observables. All plots update live.

In [ ]:
# ── imports & constants ────────────────────────────────────────────────────────
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from pathlib import Path

import sys
sys.path.insert(0, '/lustre2/nplqcd/vayyar/grid_qcd/grid-lqcd-workflow/4_analysis')
from hmc import load_run
from hmc.autocorr import gamma_method
from hmc.equilibrate import suggest_burnin

plt.rcParams.update({'figure.dpi': 110, 'font.size': 11})

BASE     = Path('/lustre2/nplqcd/vayyar/grid_qcd/runs')
ALL_OBS  = ['plaquette', 'polyakov_abs', 'dH']
OBS_LABELS = {'plaquette': 'Plaquette', 'polyakov_abs': '|Polyakov loop|', 'dH': 'dH'}

In [ ]:
# ── data loading ───────────────────────────────────────────────────────────────
def discover_runs(base):
    runs = {}
    for d in sorted(base.iterdir()):
        hmc_dir = d / 'hmc'
        if d.is_dir() and hmc_dir.is_dir() and list(hmc_dir.glob('hmc_traj*.log')):
            runs[d.name] = hmc_dir
    return runs

RUN_DIRS = discover_runs(BASE)
print('Available runs:', list(RUN_DIRS.keys()))

_cache = {}
def get_df(run_name):
    if run_name not in _cache:
        _cache[run_name] = load_run(RUN_DIRS[run_name])
    return _cache[run_name]

In [ ]:
# ── plot functions ─────────────────────────────────────────────────────────────
def available_obs(df):
    return [c for c in ALL_OBS if c in df.columns and df[c].notna().any()]

def draw(run_name, burnin, observables):
    out.clear_output(wait=True)
    df   = get_df(run_name)
    prod = df[df.traj >= burnin]

    with out:
        # --- Time series (plaquette + polyakov on shared x-axis) ---
        avail_ts = [c for c in ['plaquette', 'polyakov_abs']
                    if c in df.columns and df[c].notna().any()]
        if avail_ts:
            fig, axes = plt.subplots(len(avail_ts), 1,
                                     figsize=(10, 3 * len(avail_ts)),
                                     sharex=True)
            if len(avail_ts) == 1:
                axes = [axes]
            for ax, col in zip(axes, avail_ts):
                ax.plot(df.traj,   df[col],   'o-', ms=3, lw=0.8, color='lightsteelblue')
                ax.plot(prod.traj, prod[col],  'o-', ms=3, lw=0.8, color='steelblue', label='production')
                ax.axvline(burnin, color='red', ls='--', lw=1.5, label=f'burn-in = {burnin}')
                ax.set_ylabel(OBS_LABELS.get(col, col))
                ax.legend(fontsize=9)
                ax.grid(True, alpha=0.3)
            axes[-1].set_xlabel('Trajectory')
            acc   = prod.accepted.mean() if prod.accepted.notna().any() else None
            title = f'{run_name}   |   {len(prod)} production trajs'
            if acc is not None:
                title += f'   |   acc. rate {acc:.1%}'
            fig.suptitle(title, fontsize=12)
            plt.tight_layout()
            plt.show()

        # --- ACF ---
        results = {}
        for col in observables:
            if col not in df.columns:
                continue
            data = prod[col].dropna().values
            if len(data) >= 4:
                try:
                    results[col] = gamma_method(data)
                except Exception as e:
                    print(f'{col}: {e}')

        if results:
            fig, axes = plt.subplots(1, len(results), figsize=(5 * len(results), 4))
            if len(results) == 1:
                axes = [axes]
            for ax, col in zip(axes, results):
                res = results[col]
                ax.bar(np.arange(len(res['rho'])), res['rho'], width=0.8,
                       color='steelblue', alpha=0.7)
                ax.axhline(0, color='k', lw=0.8)
                ax.set_xlabel('Lag')
                ax.set_ylabel('rho(t)')
                ax.set_title(f"{OBS_LABELS.get(col, col)}\n"
                             f"tau_int = {res['tau_int']:.2f} +/- {res['tau_int_err']:.2f}")
                ax.grid(True, alpha=0.3)
            plt.suptitle('Autocorrelation (Gamma method)', fontsize=12)
            plt.tight_layout()
            plt.show()

        # --- dH histograms ---
        if 'dH' in df.columns:
            fig, axes = plt.subplots(1, 2, figsize=(10, 4))
            dH = prod['dH'].dropna()
            axes[0].hist(dH, bins=20, color='steelblue', alpha=0.7, edgecolor='white')
            axes[0].axvline(0, color='red', ls='--', lw=1.2)
            axes[0].set_xlabel('dH')
            axes[0].set_ylabel('Count')
            axes[0].set_title(f'dH  (mean = {dH.mean():.4f})')
            axes[0].grid(True, alpha=0.3)
            exp_dH = prod['exp_dH'].dropna()
            axes[1].hist(exp_dH, bins=20, color='coral', alpha=0.7, edgecolor='white')
            axes[1].axvline(1, color='red', ls='--', lw=1.2, label='ideal = 1')
            axes[1].set_xlabel('exp(-dH)')
            axes[1].set_ylabel('Count')
            axes[1].set_title(f'exp(-dH)  (mean = {exp_dH.mean():.4f})')
            axes[1].legend()
            axes[1].grid(True, alpha=0.3)
            plt.suptitle('Integrator quality', fontsize=12)
            plt.tight_layout()
            plt.show()

        # --- Summary table ---
        if results:
            rows = []
            for col, res in results.items():
                n = len(prod[col].dropna())
                rows.append({
                    'Observable' : OBS_LABELS.get(col, col),
                    'Mean'       : f"{res['mean']:.6f}",
                    'Stat. error': f"{res['sigma']:.6f}",
                    'tau_int'    : f"{res['tau_int']:.2f} +/- {res['tau_int_err']:.2f}",
                    'Window'     : res['window'],
                    'N_eff'      : f"{n / (2 * res['tau_int']):.1f}",
                })
            display(pd.DataFrame(rows).set_index('Observable'))

In [ ]:
# ── widgets ────────────────────────────────────────────────────────────────────
df0 = get_df(list(RUN_DIRS)[0])

run_w = widgets.Dropdown(
    options=list(RUN_DIRS),
    description='Run:',
    layout=widgets.Layout(width='260px'),
)
burnin_w = widgets.IntSlider(
    min=0, max=int(df0.traj.max()), step=5, value=0,
    description='Burn-in:',
    continuous_update=False,
    style={'description_width': '60px'},
    layout=widgets.Layout(width='400px'),
)
obs_w = widgets.SelectMultiple(
    options=available_obs(df0),
    value=available_obs(df0)[:2],
    description='ACF obs:',
    rows=3,
    layout=widgets.Layout(width='220px'),
)

out      = widgets.Output()
controls = widgets.VBox([widgets.HBox([run_w, burnin_w, obs_w])])

def _on_change(_):
    draw(run_w.value, burnin_w.value, obs_w.value)

def _on_run_change(change):
    # Update burnin and obs_w silently (detach observers) then draw once
    burnin_w.unobserve(_on_change, names='value')
    obs_w.unobserve(_on_change, names='value')
    df = get_df(change['new'])
    burnin_w.max   = int(df.traj.max())
    burnin_w.value = int(suggest_burnin(df) or df.traj.min())
    avail          = available_obs(df)
    obs_w.options  = avail
    obs_w.value    = avail[:2]
    burnin_w.observe(_on_change, names='value')
    obs_w.observe(_on_change, names='value')
    draw(change['new'], burnin_w.value, obs_w.value)

run_w.observe(_on_run_change, names='value')
burnin_w.observe(_on_change, names='value')
obs_w.observe(_on_change, names='value')

In [ ]:
# ── display (re-run this cell to refresh) ─────────────────────────────────────
display(controls, out)
draw(run_w.value, burnin_w.value, obs_w.value)